In [0]:
dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("file_name", "")

batch_id = dbutils.widgets.get("batch_id").strip()
file_name = dbutils.widgets.get("file_name").strip()

if batch_id and file_name:
    print("Batch ID:", batch_id)
    print("File name:", file_name)
else:
    print("No parameters supplied. Waiting for job parameters.")

Batch ID: BATCH_005
File name: batch_5.csv


Imports and configuration

In [0]:
# Import required libraries

from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)

from delta.tables import DeltaTable

# Catalog and schema

CATALOG = "incremental_catalog"
SCHEMA = "pipeline_schema"

# Table names

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_orders"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_orders"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_orders_summary"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.quarantine_orders"
AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_audit"
TRACKING_TABLE = f"{CATALOG}.{SCHEMA}.processed_batches"

# Volume path

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/order_data_volume/"
)

print("Imports completed.")
print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Pipeline configuration completed.")

Imports completed.
Catalog: incremental_catalog
Schema: pipeline_schema
Pipeline configuration completed.


Verify existing tables

In [0]:
# Verify existing pipeline tables

tables = [
    BRONZE_TABLE,
    SILVER_TABLE,
    GOLD_TABLE,
    QUARANTINE_TABLE,
    AUDIT_TABLE,
    TRACKING_TABLE
]

for table_name in tables:
    if spark.catalog.tableExists(table_name):
        print(f"EXISTS: {table_name}")
    else:
        print(f"MISSING: {table_name}")

EXISTS: incremental_catalog.pipeline_schema.bronze_orders
EXISTS: incremental_catalog.pipeline_schema.silver_orders
EXISTS: incremental_catalog.pipeline_schema.gold_orders_summary
EXISTS: incremental_catalog.pipeline_schema.quarantine_orders
EXISTS: incremental_catalog.pipeline_schema.pipeline_audit
EXISTS: incremental_catalog.pipeline_schema.processed_batches


Inspect existing table schemas

In [0]:
# Inspect schemas of existing pipeline tables

for table_name in tables:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print('=' * 60)

    spark.table(table_name).printSchema()


TABLE: incremental_catalog.pipeline_schema.bronze_orders
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file: string (nullable = true)


TABLE: incremental_catalog.pipeline_schema.silver_orders
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- rejection_reason: string (nullable = true)


TABLE: incremental_catalog.pipeline_schema.gold

In [0]:
# Record existing table counts

for table_name in tables:
    count = spark.table(table_name).count()
    print(f"{table_name.split('.')[-1]}: {count} records")

bronze_orders: 19 records
silver_orders: 11 records
gold_orders_summary: 3 records
quarantine_orders: 5 records
pipeline_audit: 6 records
processed_batches: 4 records


In [0]:
# Inspect existing processed batches

display(
    spark.table(TRACKING_TABLE)
    .orderBy("processed_at")
)

batch_id,source_file,processed_at,status,input_count
BATCH_003,batch_3.csv,2026-10-29T10:32:13.109Z,SUCCESS,6
BATCH_002,batch_2.csv,2026-10-29T10:32:13.109Z,SUCCESS,4
BATCH_001,batch_1.csv,2026-10-29T10:32:13.109Z,SUCCESS,5
BATCH_004,batch_4.csv,2026-10-30T05:57:45.990Z,SUCCESS,4


Create the batch status checker

In [0]:
def is_batch_processed(batch_id):

    processed_count = (
        spark.table(TRACKING_TABLE)
        .filter(
            (col("batch_id") == batch_id) &
            (col("status") == "SUCCESS")
        )
        .count()
    )

    return processed_count > 0

Test the batch status checker

In [0]:
# Test existing batch IDs

print("BATCH_001:", is_batch_processed("BATCH_001"))
print("BATCH_002:", is_batch_processed("BATCH_002"))
print("BATCH_003:", is_batch_processed("BATCH_003"))
print("BATCH_TEST_999:", is_batch_processed("BATCH_TEST_999"))

BATCH_001: True
BATCH_002: True
BATCH_003: True
BATCH_TEST_999: False


Check for partially processed batches

In [0]:
def check_partial_batch(batch_id):

    bronze_count = (
        spark.table(BRONZE_TABLE)
        .filter(col("batch_id") == batch_id)
        .count()
    )

    tracking_count = (
        spark.table(TRACKING_TABLE)
        .filter(
            (col("batch_id") == batch_id) &
            (col("status") == "SUCCESS")
        )
        .count()
    )

    if tracking_count > 0:
        return "COMPLETED"

    if bronze_count > 0:
        return "PARTIAL"

    return "NEW"

In [0]:
# Test batch status protection

print("BATCH_001:", check_partial_batch("BATCH_001"))
print("BATCH_002:", check_partial_batch("BATCH_002"))
print("BATCH_003:", check_partial_batch("BATCH_003"))
print("BATCH_TEST_999:", check_partial_batch("BATCH_TEST_999"))

BATCH_001: COMPLETED
BATCH_002: COMPLETED
BATCH_003: COMPLETED
BATCH_TEST_999: NEW


Read and inspect the incoming CSV

In [0]:
# Read the first CSV without modifying it

sample_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(VOLUME_PATH + "batch_1.csv")
)

print("CSV Columns:")
print(sample_df.columns)

print("\nCSV Schema:")
sample_df.printSchema()

print("\nSample Records:")
display(sample_df)

CSV Columns:
['order_id', 'customer_id', 'order_date', 'amount', 'status', 'source_updated_at']

CSV Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- source_updated_at: timestamp (nullable = true)


Sample Records:


order_id,customer_id,order_date,amount,status,source_updated_at
ORD1001,CUST01,2026-10-20,1250.5,Completed,2026-10-20T10:30:00.000Z
ORD1002,CUST02,2026-10-21,800.0,Pending,2026-10-21T09:15:00.000Z
ORD1003,CUST03,2026-10-22,2500.0,Completed,2026-10-22T14:00:00.000Z
ORD1004,CUST01,2026-10-23,450.75,Cancelled,2026-10-23T11:20:00.000Z
ORD1005,CUST04,2026-10-24,3200.0,Completed,2026-10-24T16:45:00.000Z


Create the batch preparation function

In [0]:

def prepare_batch(file_name, batch_id):

    file_path = VOLUME_PATH + file_name

    # 1. Read CSV as strings
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(file_path)
    )

    # 2. Standardize timestamp column name
    if "updated_at" in raw_df.columns:
        raw_df = raw_df.withColumnRenamed(
            "updated_at",
            "source_updated_at"
        )

    if "source_updated_at" not in raw_df.columns:
        raise ValueError(
            "Missing timestamp column: expected updated_at "
            "or source_updated_at"
        )

    print("Timestamp column standardized: source_updated_at")

    # 3. Convert columns safely
    prepared_df = (
        raw_df
        .withColumn(
            "order_date_parsed",
            F.expr("try_cast(order_date as date)")
        )
        .withColumn(
            "amount_parsed",
            F.expr("try_cast(amount as double)")
        )
        .withColumn(
            "source_updated_at_parsed",
            F.expr("try_cast(source_updated_at as timestamp)")
        )
    )

    # 4. Validate records
    prepared_df = prepared_df.withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",

            F.when(
                col("order_id").isNull() |
                (F.trim(col("order_id")) == ""),
                lit("Missing order_id")
            ),

            F.when(
                col("customer_id").isNull() |
                (F.trim(col("customer_id")) == ""),
                lit("Missing customer_id")
            ),

            F.when(
                col("order_date_parsed").isNull(),
                lit("Missing or invalid order_date")
            ),

            F.when(
                col("amount_parsed").isNull() |
                (col("amount_parsed") <= 0),
                lit("Amount must be greater than zero")
            ),

            F.when(
                col("status").isNull() |
                (F.trim(col("status")) == "") |
                (~F.lower(F.trim(col("status"))).isin(
                    "completed", "pending", "cancelled"
                )),
                lit("Invalid status")
            ),

            F.when(
                col("source_updated_at_parsed").isNull(),
                lit("Missing or invalid source_updated_at")
            )
        )
    )

    # 5. Add batch metadata
    prepared_df = (
        prepared_df
        .withColumn("batch_id", lit(batch_id))
        .withColumn("source_file", lit(file_name))
        .withColumn("ingestion_timestamp", current_timestamp())
    )

    # 6. Separate valid records
    valid_df = (
        prepared_df
        .filter(col("rejection_reason") == "")
        .select(
            "order_id",
            "customer_id",
            col("order_date_parsed").alias("order_date"),
            col("amount_parsed").alias("amount"),
            "status",
            col("source_updated_at_parsed").alias(
                "source_updated_at"
            ),
            "ingestion_timestamp",
            "batch_id",
            "source_file"
        )
    )

    # 7. Separate invalid records
    invalid_df = (
        prepared_df
        .filter(col("rejection_reason") != "")
        .select(
            "order_id",
            "customer_id",
            "order_date",
            col("amount_parsed").alias("amount"),
            "status",
            col("source_updated_at_parsed").alias(
                "source_updated_at"
            ),
            "order_date_parsed",
            "rejection_reason",
            "batch_id",
            "source_file"
        )
    )

    # 8. Return results
    return raw_df, valid_df, invalid_df


In [0]:
# Test batch preparation using Batch 3

raw_test, valid_test, invalid_test = prepare_batch(
    "batch_3.csv",
    "BATCH_TEST_ONLY"
)

print("Raw records:", raw_test.count())
print("Valid records:", valid_test.count())
print("Invalid records:", invalid_test.count())

print("\nValid records:")
display(valid_test)

print("\nInvalid records:")
display(invalid_test.select(
    "order_id",
    "rejection_reason"
))

Timestamp column standardized: source_updated_at
Raw records: 6
Valid records: 2
Invalid records: 4

Valid records:


order_id,customer_id,order_date,amount,status,source_updated_at,ingestion_timestamp,batch_id,source_file
ORD1008,CUST06,2026-10-27,640.0,Pending,2026-10-27T09:00:00.000Z,2026-10-31T13:37:34.499Z,BATCH_TEST_ONLY,batch_3.csv
ORD0999,CUST07,2026-10-15,1100.0,Completed,2026-10-28T08:00:00.000Z,2026-10-31T13:37:34.499Z,BATCH_TEST_ONLY,batch_3.csv



Invalid records:


order_id,rejection_reason
null,Missing order_id
ORD1010,Amount must be greater than zero
ORD1011,Invalid status
ORD1012,Missing or invalid order_date


In [0]:
# Inspect validation results

display(
    invalid_test.select(
        "order_id",
        "order_date",
        "amount",
        "status",
        "rejection_reason"
    )
)

order_id,order_date,amount,status,rejection_reason
null,2026-10-28,500.0,Pending,Missing order_id
ORD1010,2026-10-28,-75.0,Completed,Amount must be greater than zero
ORD1011,2026-10-28,350.0,UnknownStatus,Invalid status
ORD1012,not-a-date,450.0,Pending,Missing or invalid order_date


Create the Bronze ingestion function

In [0]:
def write_bronze_once(raw_df, batch_id, file_name):

    existing_count = (
        spark.table(BRONZE_TABLE)
        .filter(col("batch_id") == batch_id)
        .count()
    )

    if existing_count > 0:
        raise RuntimeError(
            f"Bronze records already exist for {batch_id}. "
            "Manual recovery is required."
        )

    bronze_df = (
        raw_df
        .withColumn(
            "order_date",
            F.expr("try_cast(order_date AS DATE)")
        )
        .withColumn(
            "amount",
            F.expr("try_cast(amount AS DOUBLE)")
        )
        .withColumn(
            "source_updated_at",
            F.expr("try_cast(source_updated_at AS TIMESTAMP)")
        )
        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )
        .withColumn("batch_id", lit(batch_id))
        .withColumn("source_file", lit(file_name))
        .select(
            "order_id",
            "customer_id",
            "order_date",
            "amount",
            "status",
            "source_updated_at",
            "ingestion_timestamp",
            "batch_id",
            "source_file"
        )
    )

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    print(f"Bronze ingestion completed for {batch_id}")

    return "WRITTEN"

Create the Quarantine ingestion function

In [0]:
def write_quarantine_once(invalid_df, batch_id):

    existing_count = (
        spark.table(QUARANTINE_TABLE)
        .filter(col("batch_id") == batch_id)
        .count()
    )

    if existing_count > 0:
        raise RuntimeError(
            f"Quarantine records already exist for {batch_id}. "
            "Manual recovery is required."
        )

    quarantine_df = (
        invalid_df
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
        .select(
            "order_id",
            "customer_id",
            "order_date",
            "amount",
            "status",
            "source_updated_at",
            "order_date_parsed",
            "rejection_reason",
            "batch_id",
            "source_file",
            "quarantined_at"
        )
    )

    (
        quarantine_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE_TABLE)
    )

    print(f"Quarantine ingestion completed for {batch_id}")

    return "WRITTEN"

Create the Silver MERGE function

In [0]:
def merge_into_silver(valid_df, batch_id):

    # Remove duplicate order IDs within the incoming batch
    window_spec = Window.partitionBy("order_id").orderBy(
        col("source_updated_at").desc(),
        col("amount").desc(),
        col("status").asc(),
        col("customer_id").asc()
    )

    deduplicated_df = (
        valid_df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    # Add the Silver-specific column
    source_df = deduplicated_df.withColumn(
        "rejection_reason",
        lit(None).cast("string")
    )

    # Merge into Silver
    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

    (
        silver_delta.alias("target")
        .merge(
            source_df.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll(
            condition=(
                "source.source_updated_at > target.source_updated_at"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Silver MERGE completed for {batch_id}")

    return "MERGED"

Create the Gold summary refresh function

In [0]:
def refresh_gold():

    gold_df = (
        spark.table(SILVER_TABLE)
        .groupBy("status")
        .agg(
            F.count("*").alias("total_orders"),
            F.round(F.sum("amount"), 2).alias("total_amount")
        )
    )

    (
        gold_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .saveAsTable(GOLD_TABLE)
    )

    print("Gold summary refreshed successfully.")

    return "REFRESHED"

Create the audit logging function

In [0]:
def record_audit(
    batch_id,
    status,
    input_count,
    accepted_count,
    rejected_count,
    start_time,
    error_message=None
):

    audit_df = spark.createDataFrame(
        [(
            batch_id,
            start_time,
            None,
            status,
            input_count,
            accepted_count,
            rejected_count,
            error_message
        )],
        schema=spark.table(AUDIT_TABLE).schema
    )

    audit_df = (
        audit_df
        .withColumn("start_time", col("start_time").cast("timestamp"))
        .withColumn("end_time", current_timestamp())
    )

    audit_delta = DeltaTable.forName(spark, AUDIT_TABLE)

    (
        audit_delta.alias("target")
        .merge(
            audit_df.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Audit record saved: {batch_id} - {status}")

Create the processed-batch tracking function

In [0]:
def record_processed_batch(batch_id, file_name, input_count):

    tracking_schema = spark.table(TRACKING_TABLE).schema

    tracking_data = [(
        batch_id,
        file_name,
        None,
        "SUCCESS",
        input_count
    )]

    tracking_df = spark.createDataFrame(
        tracking_data,
        schema=tracking_schema
    ).withColumn(
        "processed_at",
        current_timestamp()
    )

    tracking_delta = DeltaTable.forName(spark, TRACKING_TABLE)

    (
        tracking_delta.alias("target")
        .merge(
            tracking_df.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Batch tracking saved: {batch_id}")

Verify all pipeline functions are defined

In [0]:
# Verify function definitions

functions = [
    prepare_batch,
    write_bronze_once,
    write_quarantine_once,
    merge_into_silver,
    refresh_gold,
    record_audit,
    record_processed_batch,
    check_partial_batch,
    is_batch_processed
]

for function in functions:
    print(f"READY: {function.__name__}")

READY: prepare_batch
READY: write_bronze_once
READY: write_quarantine_once
READY: merge_into_silver
READY: refresh_gold
READY: record_audit
READY: record_processed_batch
READY: check_partial_batch
READY: is_batch_processed


Create the main pipeline controller

In [0]:
def run_pipeline(batch_id, file_name):

    start_time = datetime.now()

    input_count = 0
    accepted_count = 0
    rejected_count = 0

    try:
        # 1. Check batch status
        batch_status = check_partial_batch(batch_id)

        if batch_status == "COMPLETED":
            print(f"{batch_id} already completed. Skipping.")
            return "SKIPPED"

        if batch_status == "PARTIAL":
            raise RuntimeError(
                f"{batch_id} contains partial data. "
                "Manual recovery is required."
            )

        print(f"Starting pipeline: {batch_id}")

        # 2. Read and validate the CSV
        raw_df, valid_df, invalid_df = prepare_batch(
            file_name, batch_id
        )

        input_count = raw_df.count()
        accepted_count = valid_df.count()
        rejected_count = invalid_df.count()

        print("Input records:", input_count)
        print("Accepted records:", accepted_count)
        print("Rejected records:", rejected_count)

        # 3. Write raw data to Bronze
        bronze_status = write_bronze_once(
            raw_df, batch_id, file_name
        )

        if bronze_status != "WRITTEN":
            raise RuntimeError("Bronze ingestion failed.")

        # 4. Write invalid records to Quarantine
        if rejected_count > 0:
            write_quarantine_once(invalid_df, batch_id)

        # 5. Merge valid records into Silver
        if accepted_count > 0:
            merge_into_silver(valid_df, batch_id)

        # 6. Refresh Gold
        refresh_gold()

        # 7. Record successful audit
        record_audit(
            batch_id,
            "SUCCESS",
            input_count,
            accepted_count,
            rejected_count,
            start_time
        )

        # 8. Record successful tracking
        record_processed_batch(
            batch_id,
            file_name,
            input_count
        )

        print(f"Pipeline completed: {batch_id}")

        return "SUCCESS"

    except Exception as e:

        error_message = str(e)[:1000]

        print("Pipeline failed:", error_message)

        # Record failure audit when possible
        try:
            current_status = check_partial_batch(batch_id)

            if current_status != "COMPLETED":
                record_audit(
                    batch_id,
                    "FAILED",
                    input_count,
                    accepted_count,
                    rejected_count,
                    start_time,
                    error_message
                )

        except Exception as audit_error:
            print(
                "Failure audit could not be saved:",
                str(audit_error)
            )

        raise

In [0]:
# Verify the main pipeline function

print("Function name:", run_pipeline.__name__)
print("Function arguments:", run_pipeline.__code__.co_varnames[:2])
print("Pipeline function is ready.")

Function name: run_pipeline
Function arguments: ('batch_id', 'file_name')
Pipeline function is ready.


In [0]:
# Check duplicate order IDs in Silver

duplicate_orders = (
    spark.table(SILVER_TABLE)
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_count = duplicate_orders.count()

print("Duplicate order IDs:", duplicate_count)

display(duplicate_orders)

Duplicate order IDs: 0


order_id,count


In [0]:
# Calculate totals from Silver

silver_total = (
    spark.table(SILVER_TABLE)
    .agg(
        F.count("*").alias("total_orders"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .first()
)

# Calculate totals from Gold

gold_total = (
    spark.table(GOLD_TABLE)
    .agg(
        F.sum("total_orders").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_amount")
    )
    .first()
)

print("Silver orders:", silver_total["total_orders"])
print("Gold orders:", gold_total["total_orders"])

print("Silver amount:", silver_total["total_amount"])
print("Gold amount:", gold_total["total_amount"])

print(
    "Order reconciliation:",
    silver_total["total_orders"] == gold_total["total_orders"]
)

print(
    "Amount reconciliation:",
    silver_total["total_amount"] == gold_total["total_amount"]
)

Silver orders: 11
Gold orders: 11
Silver amount: 15041.24
Gold amount: 15041.24
Order reconciliation: True
Amount reconciliation: True


In [0]:
# Inspect existing quarantine records

display(
    spark.table(QUARANTINE_TABLE)
    .select(
        "order_id",
        "rejection_reason",
        "batch_id"
    )
    .orderBy("batch_id")
)

order_id,rejection_reason,batch_id
ORD1012,Missing or invalid order_date,BATCH_003
ORD1010,Amount must be greater than zero,BATCH_003
ORD1011,Invalid status,BATCH_003
null,Missing order_id,BATCH_003
ORD1015,Amount must be greater than zero,BATCH_004


In [0]:
# Check Bronze records by batch

display(
    spark.table(BRONZE_TABLE)
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy("batch_id")
)

batch_id,record_count
BATCH_001,5
BATCH_002,4
BATCH_003,6
BATCH_004,4


In [0]:
# Check Silver records by batch

display(
    spark.table(SILVER_TABLE)
    .groupBy("batch_id")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy("batch_id")
)

batch_id,record_count
BATCH_001,4
BATCH_002,2
BATCH_003,2
BATCH_004,3


In [0]:
# Inspect the updated order

display(
    spark.table(SILVER_TABLE)
    .filter(col("order_id") == "ORD1003")
)

order_id,customer_id,order_date,amount,status,source_updated_at,ingestion_timestamp,batch_id,source_file,rejection_reason
ORD1003,CUST03,2026-10-22,2900.0,Completed,2026-10-30T09:00:00.000Z,2026-10-30T05:57:32.362Z,BATCH_004,batch_4.csv,null


In [0]:
# Inspect the late-arriving order

display(
    spark.table(SILVER_TABLE)
    .filter(col("order_id") == "ORD0999")
)

order_id,customer_id,order_date,amount,status,source_updated_at,ingestion_timestamp,batch_id,source_file,rejection_reason
ORD0999,CUST07,2026-10-15,1100.0,Completed,2026-10-28T08:00:00.000Z,2026-10-29T10:21:27.499Z,BATCH_003,batch_3.csv,


In [0]:
# Inspect existing audit records

display(
    spark.table(AUDIT_TABLE)
    .orderBy("batch_id")
)

batch_id,start_time,end_time,status,input_count,accepted_count,rejected_count,error_message
BATCH_001,2026-10-29T09:29:00.000Z,2026-10-29T10:30:59.318Z,SUCCESS,5,5,0,null
BATCH_002,2026-10-29T10:02:00.000Z,2026-10-29T10:30:59.318Z,SUCCESS,4,4,0,null
BATCH_003,2026-10-29T10:20:00.000Z,2026-10-29T10:30:59.318Z,SUCCESS,6,2,4,null
BATCH_004,2026-10-30T05:57:22.743Z,2026-10-30T05:57:43.020Z,SUCCESS,4,3,1,null
BATCH_005,2026-10-31T13:34:58.701Z,2026-10-31T13:35:03.714Z,FAILED,2,2,0,"[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `source_updated_at` cannot be resolved. Did you mean one of the following? [`updated_at`, `order_date`, `order_id`, `customer_id`, `amount`]. SQLSTATE: 42703; line 1 pos 9; 'Project ['order_id, 'customer_id, 'order_date, 'amount, 'status, 'source_updated_at, 'ingestion_timestamp, 'batch_id, 'source_file] +- 'Project [unresolvedstarwithcolumns(source_file, batch_5.csv, Some(List({})))] +- 'Project [unresolvedstarwithcolumns(batch_id, BATCH_005, Some(List({})))] +- 'Project [unresolvedstarwithcolumns(ingestion_timestamp, current_timestamp(), Some(List({})))] +- 'Project [order_id#16378, customer_id#16379, order_date#16481, amount#16483, status#16382, updated_at#16383, try_cast('source_updated_at as timestamp) AS source_updated_at#16489] +- Project [order_id#16378, customer_id#16379, order_date#16481, try_cast(amount#16381 as double) AS amount#16483, status#16382, updated_"
BATCH_TEST_999,2026-10-30T07:52:46.911Z,2026-10-30T07:52:50.058Z,FAILED,0,0,0,[PATH_NOT_FOUND] Path does not exist: dbfs:/Volumes/incremental_catalog/pipeline_schema/order_data_volume/test.csv. SQLSTATE: 42K03 JVM stacktrace: org.apache.spark.sql.AnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3564) at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:1225) at scala.collection.immutable.List.flatMap(List.scala:283) at scala.collection.immutable.List.flatMap(List.scala:79) at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:1206) at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(DataSource.scala:822) at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:640) at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.$anonfun$loadV1BatchSource$3(ResolveDataSource.scala:400) at org.apache.spark.sql.c


In [0]:
# Final baseline verification

for table_name in tables:
    print(
        table_name.split(".")[-1],
        "→",
        spark.table(table_name).count()
    )

bronze_orders → 19
silver_orders → 11
gold_orders_summary → 3
quarantine_orders → 5
pipeline_audit → 6
processed_batches → 4


In [0]:
import inspect

print(inspect.signature(record_audit))
print(inspect.getsource(record_audit))

(batch_id, status, input_count, accepted_count, rejected_count, start_time, error_message=None)
def record_audit(
    batch_id,
    status,
    input_count,
    accepted_count,
    rejected_count,
    start_time,
    error_message=None
):

    audit_df = spark.createDataFrame(
        [(
            batch_id,
            start_time,
            None,
            status,
            input_count,
            accepted_count,
            rejected_count,
            error_message
        )],
        schema=spark.table(AUDIT_TABLE).schema
    )

    audit_df = (
        audit_df
        .withColumn("start_time", col("start_time").cast("timestamp"))
        .withColumn("end_time", current_timestamp())
    )

    audit_delta = DeltaTable.forName(spark, AUDIT_TABLE)

    (
        audit_delta.alias("target")
        .merge(
            audit_df.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .e

In [0]:
import inspect

print(inspect.getsource(record_processed_batch))

def record_processed_batch(batch_id, file_name, input_count):

    tracking_schema = spark.table(TRACKING_TABLE).schema

    tracking_data = [(
        batch_id,
        file_name,
        None,
        "SUCCESS",
        input_count
    )]

    tracking_df = spark.createDataFrame(
        tracking_data,
        schema=tracking_schema
    ).withColumn(
        "processed_at",
        current_timestamp()
    )

    tracking_delta = DeltaTable.forName(spark, TRACKING_TABLE)

    (
        tracking_delta.alias("target")
        .merge(
            tracking_df.alias("source"),
            "target.batch_id = source.batch_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Batch tracking saved: {batch_id}")



In [0]:
import inspect
print(inspect.getsource(write_bronze_once))

def write_bronze_once(raw_df, batch_id, file_name):

    existing_count = (
        spark.table(BRONZE_TABLE)
        .filter(col("batch_id") == batch_id)
        .count()
    )

    if existing_count > 0:
        raise RuntimeError(
            f"Bronze records already exist for {batch_id}. "
            "Manual recovery is required."
        )

    bronze_df = (
        raw_df
        .withColumn(
            "order_date",
            F.expr("try_cast(order_date AS DATE)")
        )
        .withColumn(
            "amount",
            F.expr("try_cast(amount AS DOUBLE)")
        )
        .withColumn(
            "source_updated_at",
            F.expr("try_cast(source_updated_at AS TIMESTAMP)")
        )
        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )
        .withColumn("batch_id", lit(batch_id))
        .withColumn("source_file", lit(file_name))
        .select(
            "order_id",
            "customer_id",
            "orde

In [0]:
import inspect
print(inspect.getsource(write_quarantine_once))

def write_quarantine_once(invalid_df, batch_id):

    existing_count = (
        spark.table(QUARANTINE_TABLE)
        .filter(col("batch_id") == batch_id)
        .count()
    )

    if existing_count > 0:
        raise RuntimeError(
            f"Quarantine records already exist for {batch_id}. "
            "Manual recovery is required."
        )

    quarantine_df = (
        invalid_df
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
        .select(
            "order_id",
            "customer_id",
            "order_date",
            "amount",
            "status",
            "source_updated_at",
            "order_date_parsed",
            "rejection_reason",
            "batch_id",
            "source_file",
            "quarantined_at"
        )
    )

    (
        quarantine_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE_TABLE)
    )

    print(f"Quarantine ingestion completed for {ba

In [0]:
raw_df, valid_df, invalid_df = prepare_batch(
    "batch_3.csv",
    "BATCH_TEST_ONLY"
)

print("Input records:", raw_df.count())
print("Valid records:", valid_df.count())
print("Invalid records:", invalid_df.count())

display(
    invalid_df.select(
        "order_id",
        "status",
        "rejection_reason"
    )
)

Timestamp column standardized: source_updated_at
Input records: 6
Valid records: 2
Invalid records: 4


order_id,status,rejection_reason
null,Pending,Missing order_id
ORD1010,Completed,Amount must be greater than zero
ORD1011,UnknownStatus,Invalid status
ORD1012,Pending,Missing or invalid order_date


In [0]:
for table in [
    BRONZE_TABLE,
    SILVER_TABLE,
    GOLD_TABLE,
    QUARANTINE_TABLE,
    AUDIT_TABLE,
    TRACKING_TABLE
]:
    print(table, spark.table(table).count())

incremental_catalog.pipeline_schema.bronze_orders 19
incremental_catalog.pipeline_schema.silver_orders 11
incremental_catalog.pipeline_schema.gold_orders_summary 3
incremental_catalog.pipeline_schema.quarantine_orders 5
incremental_catalog.pipeline_schema.pipeline_audit 6
incremental_catalog.pipeline_schema.processed_batches 4


In [0]:
print("BATCH_004 tracking records:",
      spark.table(TRACKING_TABLE)
      .filter(col("batch_id") == "BATCH_004")
      .count())

print("BATCH_004 Bronze records:",
      spark.table(BRONZE_TABLE)
      .filter(col("batch_id") == "BATCH_004")
      .count())

BATCH_004 tracking records: 1
BATCH_004 Bronze records: 4


In [0]:
display(
    dbutils.fs.ls(VOLUME_PATH)
)

path,name,size,modificationTime
dbfs:/Volumes/incremental_catalog/pipeline_schema/order_data_volume/batch_1.csv,batch_1.csv,376,1793259456000
dbfs:/Volumes/incremental_catalog/pipeline_schema/order_data_volume/batch_2.csv,batch_2.csv,311,1793259455000
dbfs:/Volumes/incremental_catalog/pipeline_schema/order_data_volume/batch_3.csv,batch_3.csv,428,1793259455000
dbfs:/Volumes/incremental_catalog/pipeline_schema/order_data_volume/batch_4.csv,batch_4.csv,305,1793339760000
dbfs:/Volumes/incremental_catalog/pipeline_schema/order_data_volume/batch_5.csv,batch_5.csv,176,1793347426000


In [0]:
run_pipeline(
    "BATCH_004",
    "batch_4.csv"
)

BATCH_004 already completed. Skipping.


'SKIPPED'

In [0]:
from pyspark.sql.functions import col

BRONZE_TABLE = "incremental_catalog.pipeline_schema.bronze_orders"
SILVER_TABLE = "incremental_catalog.pipeline_schema.silver_orders"
GOLD_TABLE = "incremental_catalog.pipeline_schema.gold_orders_summary"
QUARANTINE_TABLE = "incremental_catalog.pipeline_schema.quarantine_orders"
AUDIT_TABLE = "incremental_catalog.pipeline_schema.pipeline_audit"
TRACKING_TABLE = "incremental_catalog.pipeline_schema.processed_batches"

# 1. Batch 4 record counts
print("BRONZE:", spark.table(BRONZE_TABLE)
      .filter(col("batch_id") == "BATCH_004").count())

print("SILVER:", spark.table(SILVER_TABLE)
      .filter(col("batch_id") == "BATCH_004").count())

print("QUARANTINE:", spark.table(QUARANTINE_TABLE)
      .filter(col("batch_id") == "BATCH_004").count())

# 2. Verify updated order
display(
    spark.table(SILVER_TABLE)
    .filter(col("order_id") == "ORD1003")
    .select("order_id", "amount", "source_updated_at", "batch_id")
)

# 3. Audit
display(
    spark.table(AUDIT_TABLE)
    .filter(col("batch_id") == "BATCH_004")
)

# 4. Tracking
display(
    spark.table(TRACKING_TABLE)
    .filter(col("batch_id") == "BATCH_004")
)

# 5. Gold summary
display(spark.table(GOLD_TABLE))

BRONZE: 4
SILVER: 3
QUARANTINE: 1


order_id,amount,source_updated_at,batch_id
ORD1003,2900.0,2026-10-30T09:00:00.000Z,BATCH_004


batch_id,start_time,end_time,status,input_count,accepted_count,rejected_count,error_message
BATCH_004,2026-10-30T05:57:22.743Z,2026-10-30T05:57:43.020Z,SUCCESS,4,3,1,null


batch_id,source_file,processed_at,status,input_count
BATCH_004,batch_4.csv,2026-10-30T05:57:45.990Z,SUCCESS,4


status,total_orders,total_amount
Pending,4,3940.0
Cancelled,1,450.75
Completed,6,10650.49


In [0]:
if batch_id and file_name:
    result = run_pipeline(batch_id, file_name)
    dbutils.notebook.exit(result)
else:
    print("No parameters supplied. Pipeline not started.")